In [2]:
import numpy as np
import pandas as pd

# Load dataset
data = pd.DataFrame({
'Sky': ['Sunny', 'Sunny', 'Rainy', 'Sunny'],
'Temperature': ['Warm', 'Warm', 'Cold', 'Warm'],
'Humidity': ['Normal', 'High', 'High', 'High'],
'Wind': ['Strong', 'Strong', 'Strong', 'Weak'],
'EnjoySport': ['Yes', 'Yes', 'No', 'Yes']
})

print(data)

     Sky Temperature Humidity    Wind EnjoySport
0  Sunny        Warm   Normal  Strong        Yes
1  Sunny        Warm     High  Strong        Yes
2  Rainy        Cold     High  Strong         No
3  Sunny        Warm     High    Weak        Yes


In [3]:
# Attributes and target

attributes = ['Sky', 'Temperature', 'Humidity', 'Wind']
target = 'EnjoySport'

print("Attributes: ", attributes)
print("Target: ", target)

Attributes:  ['Sky', 'Temperature', 'Humidity', 'Wind']
Target:  EnjoySport


In [16]:
# Initialize S AND G

S = ['Φ'] * len(attributes) # Φ - alt + 232
G = [['?'] * len(attributes)]

print("Initial S: ", S)
print("Initial G: ", G)

Initial S:  ['Φ', 'Φ', 'Φ', 'Φ']
Initial G:  [['?', '?', '?', '?']]


In [17]:
# Covers function

def covers(hypothesis, example):
    for h, e in zip(hypothesis, example):
        if h != '?' and h != 'Φ' and h != e:
            return False
        return True
    


In [18]:
# Generalize S

def generalize_S(S, example):
    new_S = S.copy()
    for i in range(len(S)):
        if S[i] == 'Φ':
            new_S[i] = example[i]
        elif S[i] != example[i]: 
            new_S[i] = '?'
        return new_S

In [19]:
# Possible Values

domains = [
    ['Sunny', 'Rainy'],
    ['Warm', 'Cold'],
    ['Normal', 'High'],
    ['Strong', 'Weak']
]

In [22]:
# Specialize G

def Specialize_G(G, example, S, domains):
    new_G = []

    for g in G:
        # Check whether G covers the negative example
        if covers(g, example):

            # Try every attribute
            for i in range(len(g)):

                if g[i] == '?':
                    # Try possible values
                    for value in domains[i]:

                        # Values must reject the negative example
                        if value != example[i]:
                            hypothesis = g.copy()
                            hypothesis[i] = value

                            # Check consistency with S
                            valid = True

                            for j in range(len(S)):
                                if S[j] != 'Φ' and hypothesis[j] != '?':
                                    if hypothesis[j] != S[j]:
                                        valid = False
                                        break

                            # Add hypothesis if it is consistent
                            if valid:
                                new_G.append(hypothesis)

    return new_G



In [23]:
# Candidate Elimination

for index, row in data.iterrows():
    example = row[attributes].tolist()
    label = row[target]
    print("\n================================")
    print("Example", index + 1)
    print("Example: ", example)
    print("Target: ", label)
    
    # Positive Example
    if label == 'Yes':
        print("Postive example")
        # Remove G hypothesis that don't cover the positive example
        
        G = [ g for g in G
            if covers(g, example)
            ]
        # Generalize s
        S = generalize_S(S, example)
        
    # negative example
    else :
        print("Negative example")
        # Specialize G
        G = Specialize_G(G, example, S, domains)
            
     # Display current boundaries
        
    print("S = ", S)
    print("G = ", G)


Example 1
Example:  ['Sunny', 'Warm', 'Normal', 'Strong']
Target:  Yes
Postive example
S =  ['Sunny', 'Φ', 'Φ', 'Φ']
G =  [['?', '?', '?', '?']]

Example 2
Example:  ['Sunny', 'Warm', 'High', 'Strong']
Target:  Yes
Postive example
S =  ['Sunny', 'Φ', 'Φ', 'Φ']
G =  [['?', '?', '?', '?']]

Example 3
Example:  ['Rainy', 'Cold', 'High', 'Strong']
Target:  No
Negative example
S =  ['Sunny', 'Φ', 'Φ', 'Φ']
G =  [['Sunny', '?', '?', '?'], ['?', 'Warm', '?', '?'], ['?', '?', 'Normal', '?'], ['?', '?', '?', 'Weak']]

Example 4
Example:  ['Sunny', 'Warm', 'High', 'Weak']
Target:  Yes
Postive example
S =  ['Sunny', 'Φ', 'Φ', 'Φ']
G =  [['Sunny', '?', '?', '?'], ['?', 'Warm', '?', '?'], ['?', '?', 'Normal', '?'], ['?', '?', '?', 'Weak']]
